# Calendar Dimension Table

Builds the standard calendar dimension `adb_caresync.silver.calendar` with one row per day from 1940-01-01 to 2030-12-31, including year, quarter, month, week, day and flag attributes.

In [0]:
%sql
CREATE OR REPLACE TABLE adb_caresync.silver.calendar
COMMENT 'Standard calendar dimension: one row per day from 1940-01-01 to 2030-12-31'
AS
WITH dates AS (
  SELECT explode(sequence(DATE'1940-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS d
)
SELECT
  CAST(date_format(d, 'yyyyMMdd') AS INT)                 AS date_key,
  d                                                       AS date,
  -- Year
  year(d)                                                 AS year,
  trunc(d, 'YEAR')                                        AS first_day_of_year,
  date_sub(add_months(trunc(d, 'YEAR'), 12), 1)           AS last_day_of_year,
  CASE WHEN (year(d) % 4 = 0 AND year(d) % 100 <> 0) OR year(d) % 400 = 0
       THEN true ELSE false END                           AS is_leap_year,
  -- Quarter
  quarter(d)                                              AS quarter,
  concat('Q', quarter(d))                                 AS quarter_name,
  concat(year(d), '-Q', quarter(d))                       AS year_quarter,
  trunc(d, 'QUARTER')                                     AS first_day_of_quarter,
  date_sub(add_months(trunc(d, 'QUARTER'), 3), 1)         AS last_day_of_quarter,
  datediff(d, trunc(d, 'QUARTER')) + 1                    AS day_of_quarter,
  -- Month
  month(d)                                                AS month,
  date_format(d, 'MMMM')                                  AS month_name,
  date_format(d, 'MMM')                                   AS month_short_name,
  date_format(d, 'yyyy-MM')                               AS year_month,
  CAST(date_format(d, 'yyyyMM') AS INT)                   AS year_month_key,
  trunc(d, 'MONTH')                                       AS first_day_of_month,
  last_day(d)                                             AS last_day_of_month,
  day(last_day(d))                                        AS days_in_month,
  -- Week (ISO 8601: weeks start Monday)
  weekofyear(d)                                           AS iso_week_of_year,
  extract(YEAROFWEEK FROM d)                              AS iso_year,
  date_sub(d, weekday(d))                                 AS week_start_date,
  date_add(date_sub(d, weekday(d)), 6)                    AS week_end_date,
  -- Day
  day(d)                                                  AS day_of_month,
  dayofyear(d)                                            AS day_of_year,
  weekday(d) + 1                                          AS day_of_week,       -- 1 = Monday ... 7 = Sunday
  date_format(d, 'EEEE')                                  AS day_name,
  date_format(d, 'EEE')                                   AS day_short_name,
  -- Flags
  weekday(d) >= 5                                         AS is_weekend,
  weekday(d) < 5                                          AS is_weekday,
  d = trunc(d, 'MONTH')                                   AS is_month_start,
  d = last_day(d)                                         AS is_month_end,
  d = trunc(d, 'QUARTER')                                 AS is_quarter_start,
  d = date_sub(add_months(trunc(d, 'QUARTER'), 3), 1)     AS is_quarter_end,
  d = trunc(d, 'YEAR')                                    AS is_year_start,
  d = date_sub(add_months(trunc(d, 'YEAR'), 12), 1)       AS is_year_end
FROM dates
ORDER BY d;

In [0]:
%sql
SELECT
  count(*)                                   AS total_rows,
  count(DISTINCT date)                       AS distinct_dates,
  min(date)                                  AS min_date,
  max(date)                                  AS max_date,
  datediff(max(date), min(date)) + 1         AS expected_rows,
  sum(CASE WHEN is_leap_year AND month = 2 AND day_of_month = 29 THEN 1 ELSE 0 END) AS leap_days
FROM adb_caresync.silver.calendar;

In [0]:
%sql
SELECT * FROM adb_caresync.silver.calendar
WHERE date BETWEEN DATE'2024-12-28' AND DATE'2025-01-03'
ORDER BY date;